# 🏠 EstateIQ Pro — Exploratory Data Analysis & Insights
### End-to-End Housing Market Analysis for King County, WA

---

## 1. Business Problem & Executive Framing
Real estate valuation often relies on manual appraisals, broker gut-feeling, or outdated comps. Mispricing listings results in prolonged market exposure (30-90+ days) and equity loss.

**EDA Objectives:**
- Evaluate target distribution (`price`) and test log transformation necessity.
- Identify core price drivers (`sqft_living`, `grade`, `zipcode`, `waterfront`).
- Discover feature interaction effects and geospatial clustering to inform the production pipeline.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_processed_data

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)

df = load_processed_data()
print(f'Cleaned dataset records: {len(df):,}')
df.head()

## 2. Univariate Analysis
### 2.1 Target Variable (`price`) Distribution
House prices in King County range from entry-level condominiums to multi-million dollar estates on Lake Washington. Let's compare raw price distribution with log-transformed price.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['price'] / 1000, kde=True, ax=axes[0], color='#2b6cb0', bins=40)
axes[0].set_title('Raw Price Distribution ($k)', fontweight='bold')
axes[0].set_xlabel('Price ($k)')

sns.histplot(np.log1p(df['price']), kde=True, ax=axes[1], color='#2c7a7b', bins=40)
axes[1].set_title('Log-Transformed Price (np.log1p)', fontweight='bold')
axes[1].set_xlabel('Log(Price)')
plt.tight_layout()
plt.show()

print(f"Raw Price Skewness: {df['price'].skew():.2f}")
print(f"Log(Price) Skewness: {np.log1p(df['price']).skew():.2f}")

**Business Takeaway:** Log-transforming the target reduces skewness from high positive skew to near-normal distribution. This stabilizes gradient descent and minimizes relative percentage error (MAPE) across median homes.

### 2.2 Construction Grade & Size Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x=df['grade'], y=df['price'] / 1000, ax=axes[0], palette='viridis')
axes[0].set_title('Price by Construction Grade (1-13)', fontweight='bold')
axes[0].set_ylabel('Price ($k)')

sns.histplot(df['sqft_living'], kde=True, ax=axes[1], color='#4a5568', bins=40)
axes[1].set_title('Living Area Distribution (sqft)', fontweight='bold')
axes[1].set_xlabel('Sqft Living')
plt.tight_layout()
plt.show()

## 3. Bivariate & Geospatial Valuation Patterns
### 3.1 Price vs. Living Space & Waterfront Premium

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x='sqft_living',
    y=df['price'] / 1000,
    hue='waterfront',
    palette={0: '#3182ce', 1: '#e53e3e'},
    alpha=0.6,
    s=25
)
plt.title('Price vs. Living Area (Waterfront Premium)', fontsize=13, fontweight='bold')
plt.xlabel('Living Area (sqft)')
plt.ylabel('Price ($k)')
plt.legend(title='Waterfront', labels=['Standard Listing', 'Waterfront Estate'])
plt.tight_layout()
plt.show()

### 3.2 Geospatial Valuation Clusters (Seattle Metro Area)

In [ ]:
plt.figure(figsize=(11, 8))
scatter = plt.scatter(
    df['long'],
    df['lat'],
    c=np.log1p(df['price']),
    cmap='plasma',
    alpha=0.4,
    s=15
)
cbar = plt.colorbar(scatter)
cbar.set_label('Log(Price)', fontsize=11)
plt.title('Geospatial Property Price Heatmap in King County', fontsize=13, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.show()

## 4. Feature Correlation Matrix

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['id'], errors='ignore')
corr = numeric_df.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap='coolwarm', annot=False, fmt='.2f', cbar=True)
plt.title('Correlation Heatmap across Housing Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

top_corr = corr['price'].sort_values(ascending=False)
print('Top Positive Correlations with Price:')
print(top_corr.head(8))

## 5. Feature Engineering Blueprint & Takeaways
1. **Interaction Features:** The strong synergy between `sqft_living` and `grade` necessitates creating `sqft_living * grade` to capture luxury scaling.
2. **Location Encoding:** Zipcode and geographic coordinates form dense price tiers. One-hot encoding zipcodes alongside lat/long coordinates gives gradient boosting the granularity needed.
3. **Temporal Features:** `house_age = sale_year - yr_built` and `years_since_renovation` cleanly represent property lifecycle without raw timestamp noise.
